# Занятие 1. Пайплайн инвентаризации языковых датасетов

**Цель практики:** не OCR и не ASR, а стартовая карта проекта: какие языки берем в поле зрения и какие открытые данные уже можно найти.

В этой тетрадке мы собираем первоначальную информацию обычным воспроизводимым пайплайном:

1. берет редактируемый seed list основных живых языков народов России без диалектального уровня;
2. показывает, как выглядит ответ OPUS и какие поля из него достаем;
3. показывает, как выглядит карточка/ответ Hugging Face Datasets и какие поля из него достаем;
4. проверяет OPUS API на параллельные данные с русским;
5. проверяет Hugging Face Datasets как каталог опубликованных корпусов;
6. собирает таблицу, которую можно открыть в Google Sheets и дальше править руками.

Готовый снапшот этой таблицы уже создан в Google Sheets: https://docs.google.com/spreadsheets/d/1Qfr6JCB5CF-NLwQBODStqfhesrYw9tIVh2s_A6cg0d8

In [ ]:
!pip -q install pandas requests openpyxl beautifulsoup4 langgraph

In [ ]:
import os, re, json, textwrap, math, statistics, random, io, time
from pathlib import Path
import pandas as pd
import numpy as np
import requests

DATA_DIR = Path('/content/lowres_lab')
DATA_DIR.mkdir(exist_ok=True)

def show_df(df, n=10):
    display(df.head(n))

def save_artifact(name, obj):
    path = DATA_DIR / name
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(path, index=False)
    else:
        path.write_text(str(obj), encoding='utf-8')
    print('saved:', path)

from datetime import datetime, timezone
import re
from typing import Any, Dict, List, TypedDict
from urllib.parse import parse_qs, quote_plus, unquote, urljoin, urlparse
from bs4 import BeautifulSoup
from langgraph.graph import StateGraph, END

## 1. Seed list языков

Это не “истина навсегда”, а стартовая рабочая рамка для курса. Ее надо обсуждать и уточнять: какие языки добавить, где объединять варианты, где наоборот нельзя смешивать разные языковые сообщества.

In [ ]:
LANGUAGES = [{'language_ru': 'татарский', 'language_en': 'Tatar', 'family': 'Тюркская', 'branch': 'кыпчакская', 'iso639_3': 'tat', 'opus_code': 'tt'}, {'language_ru': 'башкирский', 'language_en': 'Bashkir', 'family': 'Тюркская', 'branch': 'кыпчакская', 'iso639_3': 'bak', 'opus_code': 'ba'}, {'language_ru': 'чувашский', 'language_en': 'Chuvash', 'family': 'Тюркская', 'branch': 'огурская', 'iso639_3': 'chv', 'opus_code': 'chv'}, {'language_ru': 'якутский / саха', 'language_en': 'Sakha / Yakut', 'family': 'Тюркская', 'branch': 'сибирская', 'iso639_3': 'sah', 'opus_code': 'sah'}, {'language_ru': 'тувинский', 'language_en': 'Tuvan', 'family': 'Тюркская', 'branch': 'сибирская', 'iso639_3': 'tyv', 'opus_code': 'tyv'}, {'language_ru': 'хакасский', 'language_en': 'Khakas', 'family': 'Тюркская', 'branch': 'сибирская', 'iso639_3': 'kjh', 'opus_code': 'kjh'}, {'language_ru': 'алтайский', 'language_en': 'Altai', 'family': 'Тюркская', 'branch': 'сибирская', 'iso639_3': 'alt', 'opus_code': 'alt'}, {'language_ru': 'кумыкский', 'language_en': 'Kumyk', 'family': 'Тюркская', 'branch': 'кыпчакская', 'iso639_3': 'kum', 'opus_code': 'kum'}, {'language_ru': 'карачаево-балкарский', 'language_en': 'Karachay-Balkar', 'family': 'Тюркская', 'branch': 'кыпчакская', 'iso639_3': 'krc', 'opus_code': 'krc'}, {'language_ru': 'ногайский', 'language_en': 'Nogai', 'family': 'Тюркская', 'branch': 'кыпчакская', 'iso639_3': 'nog', 'opus_code': 'nog'}, {'language_ru': 'крымскотатарский', 'language_en': 'Crimean Tatar', 'family': 'Тюркская', 'branch': 'кыпчакско-огузская', 'iso639_3': 'crh', 'opus_code': 'crh'}, {'language_ru': 'удмуртский', 'language_en': 'Udmurt', 'family': 'Уральская', 'branch': 'пермская', 'iso639_3': 'udm', 'opus_code': 'udm'}, {'language_ru': 'коми-зырянский', 'language_en': 'Komi-Zyrian', 'family': 'Уральская', 'branch': 'пермская', 'iso639_3': 'kpv', 'opus_code': 'kpv'}, {'language_ru': 'коми-пермяцкий', 'language_en': 'Komi-Permyak', 'family': 'Уральская', 'branch': 'пермская', 'iso639_3': 'koi', 'opus_code': 'koi'}, {'language_ru': 'эрзянский', 'language_en': 'Erzya', 'family': 'Уральская', 'branch': 'мордовская', 'iso639_3': 'myv', 'opus_code': 'myv'}, {'language_ru': 'мокшанский', 'language_en': 'Moksha', 'family': 'Уральская', 'branch': 'мордовская', 'iso639_3': 'mdf', 'opus_code': 'mdf'}, {'language_ru': 'марийский луговой', 'language_en': 'Meadow Mari', 'family': 'Уральская', 'branch': 'марийская', 'iso639_3': 'mhr', 'opus_code': 'mhr'}, {'language_ru': 'марийский горный', 'language_en': 'Hill Mari', 'family': 'Уральская', 'branch': 'марийская', 'iso639_3': 'mrj', 'opus_code': 'mrj'}, {'language_ru': 'карельский', 'language_en': 'Karelian', 'family': 'Уральская', 'branch': 'прибалтийско-финская', 'iso639_3': 'krl', 'opus_code': 'krl'}, {'language_ru': 'вепсский', 'language_en': 'Veps', 'family': 'Уральская', 'branch': 'прибалтийско-финская', 'iso639_3': 'vep', 'opus_code': 'vep'}, {'language_ru': 'хантыйский', 'language_en': 'Khanty', 'family': 'Уральская', 'branch': 'угорская', 'iso639_3': 'kca', 'opus_code': None}, {'language_ru': 'мансийский', 'language_en': 'Mansi', 'family': 'Уральская', 'branch': 'угорская', 'iso639_3': 'mns', 'opus_code': 'mns'}, {'language_ru': 'ненецкий', 'language_en': 'Nenets', 'family': 'Уральская', 'branch': 'самодийская', 'iso639_3': 'yrk', 'opus_code': 'yrk'}, {'language_ru': 'чеченский', 'language_en': 'Chechen', 'family': 'Северокавказская', 'branch': 'нахская', 'iso639_3': 'che', 'opus_code': 'ce'}, {'language_ru': 'ингушский', 'language_en': 'Ingush', 'family': 'Северокавказская', 'branch': 'нахская', 'iso639_3': 'inh', 'opus_code': 'inh'}, {'language_ru': 'аварский', 'language_en': 'Avar', 'family': 'Северокавказская', 'branch': 'нахско-дагестанская', 'iso639_3': 'ava', 'opus_code': 'av'}, {'language_ru': 'даргинский', 'language_en': 'Dargwa', 'family': 'Северокавказская', 'branch': 'нахско-дагестанская', 'iso639_3': 'dar', 'opus_code': 'dar'}, {'language_ru': 'лезгинский', 'language_en': 'Lezgian', 'family': 'Северокавказская', 'branch': 'нахско-дагестанская', 'iso639_3': 'lez', 'opus_code': 'lez'}, {'language_ru': 'лакский', 'language_en': 'Lak', 'family': 'Северокавказская', 'branch': 'нахско-дагестанская', 'iso639_3': 'lbe', 'opus_code': 'lbe'}, {'language_ru': 'рутульский', 'language_en': 'Rutul', 'family': 'Северокавказская', 'branch': 'нахско-дагестанская', 'iso639_3': 'rut', 'opus_code': 'rut'}, {'language_ru': 'адыгейский', 'language_en': 'Adyghe', 'family': 'Северокавказская', 'branch': 'абхазо-адыгская', 'iso639_3': 'ady', 'opus_code': 'ady'}, {'language_ru': 'кабардино-черкесский', 'language_en': 'Kabardian', 'family': 'Северокавказская', 'branch': 'абхазо-адыгская', 'iso639_3': 'kbd', 'opus_code': 'kbd'}, {'language_ru': 'абазинский', 'language_en': 'Abaza', 'family': 'Северокавказская', 'branch': 'абхазо-адыгская', 'iso639_3': 'abq', 'opus_code': None}, {'language_ru': 'бурятский', 'language_en': 'Buryat', 'family': 'Монгольская', 'branch': 'монгольская', 'iso639_3': 'bxr', 'opus_code': 'bxr'}, {'language_ru': 'калмыцкий', 'language_en': 'Kalmyk', 'family': 'Монгольская', 'branch': 'ойратская', 'iso639_3': 'xal', 'opus_code': 'xal'}, {'language_ru': 'эвенкийский', 'language_en': 'Evenki', 'family': 'Тунгусо-маньчжурская', 'branch': 'тунгусская', 'iso639_3': 'evn', 'opus_code': 'evn'}, {'language_ru': 'нанайский', 'language_en': 'Nanai', 'family': 'Тунгусо-маньчжурская', 'branch': 'тунгусская', 'iso639_3': 'gld', 'opus_code': 'gld'}, {'language_ru': 'нивхский', 'language_en': 'Nivkh', 'family': 'изолят / палеоазиатская группа', 'branch': 'нивхская', 'iso639_3': 'niv', 'opus_code': None}, {'language_ru': 'чукотский', 'language_en': 'Chukchi', 'family': 'чукотско-камчатская', 'branch': 'чукотская', 'iso639_3': 'ckt', 'opus_code': None}, {'language_ru': 'корякский', 'language_en': 'Koryak', 'family': 'чукотско-камчатская', 'branch': 'чукотская', 'iso639_3': 'kpy', 'opus_code': None}, {'language_ru': 'алеутский', 'language_en': 'Aleut', 'family': 'эскимосско-алеутская', 'branch': 'алеутская', 'iso639_3': 'ale', 'opus_code': 'ale'}, {'language_ru': 'эскимосский / юпик', 'language_en': 'Yupik', 'family': 'эскимосско-алеутская', 'branch': 'эскимосская', 'iso639_3': 'ess', 'opus_code': None}]

seed_df = pd.DataFrame(LANGUAGES)
display(seed_df.groupby(['family', 'branch']).size().reset_index(name='languages'))
display(seed_df.head(12))

## 2. Источники пайплайна: OPUS API и Hugging Face API

Здесь мы работаем со структурированными источниками готовых датасетов. Это важное ограничение: API дают воспроизводимые поля и ссылки, а веб-поиск дает только кандидатов, которые потом нужно проверять человеком.

В этом занятии мы не используем Wikipedia: это хороший источник текстовых данных, но не каталог готовых датасетов. Сейчас нас интересует именно инвентаризация уже опубликованных датасетов и корпусов.

In [ ]:
OPUS_API = 'https://opus.nlpl.eu/opusapi'
HF_DATASETS_API = 'https://huggingface.co/api/datasets'

def api_get(url, params, attempts=3, timeout=30):
    """Загружает JSON из публичного API с повторами и паузами при rate limit."""
    last_error = None
    for attempt in range(attempts):
        try:
            r = requests.get(url, params=params, timeout=timeout, headers={'User-Agent': 'lowres-course-dataset-scout/1.0'})
            if r.status_code == 429 and attempt < attempts - 1:
                wait = int(r.headers.get('Retry-After', 2 + attempt * 2))
                print('rate limit, wait', wait, 'sec')
                time.sleep(wait)
                continue
            r.raise_for_status()
            return r.json()
        except Exception as exc:
            last_error = exc
            if attempt < attempts - 1:
                time.sleep(1 + attempt * 2)
    raise last_error

def as_int(value):
    """Преобразует числовые поля OPUS в int, считая пустые значения нулем."""
    if value in ('', None):
        return 0
    return int(value)

def show_json_fragment(obj, keys=None, limit=1600):
    """Печатает небольшой фрагмент JSON, чтобы глазами увидеть форму ответа API."""
    if keys and isinstance(obj, dict):
        obj = {key: obj.get(key) for key in keys}
    text = json.dumps(obj, ensure_ascii=False, indent=2)
    print(text[:limit] + ('\n...' if len(text) > limit else ''))

def opus_pair_page_url(source, target, corpus='Tatoeba'):
    """Собирает ссылку на человеческую страницу OPUS для корпуса и языковой пары."""
    return f'https://opus.nlpl.eu/datasets/{corpus}?hi={source}&pair={target}'

def opus_pair_api_url(source, target):
    """Собирает ссылку на API-запрос OPUS для языковой пары."""
    return f'{OPUS_API}?source={source}&target={target}&preprocessing=xml&version=latest'

def hf_dataset_page_url(dataset_id):
    """Собирает ссылку на карточку датасета на Hugging Face."""
    return f'https://huggingface.co/datasets/{dataset_id}'

def hf_dataset_api_url(dataset_id):
    """Собирает ссылку на API-ответ Hugging Face по одному датасету."""
    return f'{HF_DATASETS_API}/{dataset_id}'

def dataset_search_text(dataset):
    """Склеивает metadata HF-датасета в текст для LLM-классификации."""
    tags = dataset.get('tags') or []
    return ' '.join([
        str(dataset.get('id', '')),
        str(dataset.get('description', '') or ''),
        ' '.join(tags),
    ]).lower()

def hf_search_datasets(params, limit=50):
    """Возвращает список датасетов Hugging Face по одному API-запросу."""
    query = dict(params)
    query['limit'] = limit
    data = api_get(HF_DATASETS_API, query, attempts=2, timeout=20)
    return data if isinstance(data, list) else []

def hf_datasets_for_language(language, limit_per_query=50):
    """Ищет все видимые через API HF-кандидаты для одного языка.

    Это не скачивает сами датасеты, а собирает карточки/метаданные из каталога.
    Основной критерий здесь только HF-теги `language:*`; текст карточки не используем
    как доказательство языка.
    """
    tags = {
        f"language:{language.get('opus_code')}" if language.get('opus_code') else '',
        f"language:{language.get('iso639_3')}" if language.get('iso639_3') else '',
    }
    tags.discard('')
    seen = {}
    errors = []

    for tag in sorted(tags):
        try:
            for dataset in hf_search_datasets({'filter': tag}, limit=limit_per_query):
                if dataset.get('id'):
                    seen[dataset['id']] = dataset
        except Exception as exc:
            errors.append({'query_type': 'filter', 'query': tag, 'error': str(exc)})
    return list(seen.values()), errors

def hf_datasets_for_pair(left_language, right_language, limit_per_query=50):
    """Собирает HF-кандидаты для языковой пары без текстовой псевдопроверки.

    У Hugging Face нет универсального надежного фильтра “дай все датасеты ровно
    для пары X-Y”. Поэтому берем датасеты с тегами обоих языков и добавляем
    результаты прямого pair-search. Дальше классификацию делает LLM.
    """
    seen = {}
    errors = []
    for language in [left_language, right_language]:
        datasets, language_errors = hf_datasets_for_language(language, limit_per_query=limit_per_query)
        errors.extend(language_errors)
        for dataset in datasets:
            seen[dataset['id']] = dataset

    pair_terms = [
        f"{left_language.get('language_en')} {right_language.get('language_en')}",
        f"{right_language.get('language_en')} {left_language.get('language_en')}",
        f"{left_language.get('iso639_3')}-{right_language.get('iso639_3')}",
        f"{right_language.get('iso639_3')}-{left_language.get('iso639_3')}",
        f"{left_language.get('opus_code')}-{right_language.get('opus_code')}",
        f"{right_language.get('opus_code')}-{left_language.get('opus_code')}",
    ]
    for term in [term for term in pair_terms if 'None' not in term]:
        try:
            for dataset in hf_search_datasets({'search': term}, limit=limit_per_query):
                if dataset.get('id'):
                    seen[dataset['id']] = dataset
        except Exception as exc:
            errors.append({'query_type': 'pair_search', 'query': term, 'error': str(exc)})

    left_tags = {
        f"language:{left_language.get('opus_code')}" if left_language.get('opus_code') else '',
        f"language:{left_language.get('iso639_3')}" if left_language.get('iso639_3') else '',
    }
    right_tags = {
        f"language:{right_language.get('opus_code')}" if right_language.get('opus_code') else '',
        f"language:{right_language.get('iso639_3')}" if right_language.get('iso639_3') else '',
    }
    left_tags.discard('')
    right_tags.discard('')

    pair_candidates = []
    for dataset in seen.values():
        tags = set(dataset.get('tags') or [])
        has_both_language_tags = bool(tags & left_tags) and bool(tags & right_tags)
        came_from_pair_search = any(term.lower() in dataset_search_text(dataset) for term in pair_terms if 'None' not in term)
        if has_both_language_tags or came_from_pair_search:
            pair_candidates.append(dataset)
    return pair_candidates, errors

def hf_dataset_brief_table(datasets):
    """Делает компактную таблицу из списка HF dataset API objects."""
    rows = []
    for dataset in datasets:
        tags = dataset.get('tags') or []
        rows.append({
            'id': dataset.get('id'),
            'downloads': dataset.get('downloads'),
            'likes': dataset.get('likes'),
            'language_tags': '; '.join(tag for tag in tags if tag.startswith('language:')),
            'size_categories': '; '.join(tag.replace('size_categories:', '') for tag in tags if tag.startswith('size_categories:')),
            'url': hf_dataset_page_url(dataset.get('id', '')),
        })
    return pd.DataFrame(rows)

## 3. Пример OPUS: страница пары, API-ответ и извлекаемые поля

Возьмем пару `ru-udm`: русский и удмуртский. У OPUS есть человеческие страницы корпусов и API. Для пайплайна важнее API, но страница нужна, чтобы человек мог быстро открыть источник и проверить контекст: корпус, лицензию, форматы скачивания, предупреждения OPUS.

In [ ]:
OPUS_EXAMPLE = {
    'source': 'ru',
    'target': 'udm',
    'human_page': opus_pair_page_url('ru', 'udm', corpus='Tatoeba'),
    'api_url': opus_pair_api_url('ru', 'udm'),
}
OPUS_EXAMPLE

In [ ]:
try:
    opus_raw = api_get(OPUS_API, {
        'source': OPUS_EXAMPLE['source'],
        'target': OPUS_EXAMPLE['target'],
        'preprocessing': 'xml',
        'version': 'latest',
    }, attempts=1, timeout=8)
    opus_raw_source = 'live OPUS API'
except Exception as exc:
    print('OPUS API сейчас не ответил:', exc)
    opus_raw = {'corpora': [], 'error': str(exc)}
    opus_raw_source = 'OPUS error: empty result, continue'

print('Источник примера:', opus_raw_source)
print('Страница пары/корпуса для человека:', OPUS_EXAMPLE['human_page'])
print('API URL для пайплайна:', OPUS_EXAMPLE['api_url'])
show_json_fragment(opus_raw, keys=['corpora'], limit=2200)

In [ ]:
opus_rows = pd.DataFrame(opus_raw.get('corpora', []))
expected_opus_columns = [
    'corpus',
    'source',
    'target',
    'alignment_pairs',
    'documents',
    'preprocessing',
    'version',
]
for column in expected_opus_columns:
    if column not in opus_rows.columns:
        opus_rows[column] = ''
opus_fields_we_extract = opus_rows[expected_opus_columns].copy()
display(opus_fields_we_extract)

print('Что пайплайн кладет в итоговую таблицу:')
display(pd.DataFrame([{
    'opus_ru_parallel_pairs': opus_fields_we_extract['alignment_pairs'].map(as_int).sum(),
    'opus_ru_parallel_documents': opus_fields_we_extract['documents'].map(as_int).sum(),
    'opus_ru_parallel_corpora': '; '.join(
        f"{row.corpus} ({row.alignment_pairs})"
        for row in opus_fields_we_extract.itertuples()
    ),
    'parallel_with_russian_source': OPUS_EXAMPLE['api_url'],
}]))

## 4. Пример Hugging Face: карточка датасета, API-ответ и извлекаемые поля

На Hugging Face у каждого датасета есть страница-карточка и API-ответ. Страница нужна человеку: посмотреть README, лицензию, файлы, ограничения доступа. API нужен пайплайну: собрать id, теги языка, размер, число примеров, downloads и признаки параллельности.

In [ ]:
HF_EXAMPLE_ID = 'udmurtNLP/flores-250-rus-udm'
HF_EXAMPLE = {
    'dataset_id': HF_EXAMPLE_ID,
    'human_page': hf_dataset_page_url(HF_EXAMPLE_ID),
    'api_url': hf_dataset_api_url(HF_EXAMPLE_ID),
}
HF_EXAMPLE

In [ ]:
hf_raw = api_get(HF_DATASETS_API + '/' + HF_EXAMPLE_ID, {}, attempts=2, timeout=20)

print('Страница датасета для человека:', HF_EXAMPLE['human_page'])
print('API URL для пайплайна:', HF_EXAMPLE['api_url'])
show_json_fragment(hf_raw, keys=['id', 'tags', 'downloads', 'likes', 'cardData', 'siblings'], limit=2600)

In [ ]:
card_data = hf_raw.get('cardData') or {}
dataset_info = card_data.get('dataset_info') or {}
splits = dataset_info.get('splits') or []
features = dataset_info.get('features') or []

hf_fields_we_extract = {
    'hf_dataset_id': hf_raw.get('id'),
    'hf_page': HF_EXAMPLE['human_page'],
    'hf_downloads': hf_raw.get('downloads'),
    'hf_likes': hf_raw.get('likes'),
    'hf_language_tags': '; '.join(tag for tag in hf_raw.get('tags', []) if tag.startswith('language:')),
    'hf_size_categories': '; '.join(tag.replace('size_categories:', '') for tag in hf_raw.get('tags', []) if tag.startswith('size_categories:')),
    'hf_splits': '; '.join(f"{s.get('name')} ({s.get('num_examples')} examples)" for s in splits),
    'hf_features': '; '.join(f"{f.get('name')}:{f.get('dtype')}" for f in features),
    'hf_files': '; '.join(s.get('rfilename', '') for s in hf_raw.get('siblings', [])[:5]),
}
display(pd.DataFrame([hf_fields_we_extract]).T.rename(columns={0: 'value'}))

### 4.1. Hugging Face API: все кандидаты по языку и по языковой паре

Да, через API можно собрать не только одну карточку, а список датасетов-кандидатов для конкретного языка. Для языковой пары сложнее: у HF нет одного надежного фильтра “ровно пара ru-udm”, поэтому мы комбинируем:

- `filter=language:<code>` для каждого языка;
- `search=<название языка>` и `search=<код-код>`;
- постфильтрацию по тегам и тексту карточки.

Это не скачивает все данные. Это собирает каталог кандидатов, которые потом можно открыть, скачать или отправить на human review.

In [ ]:
UDMURT = {'language_ru': 'удмуртский', 'language_en': 'Udmurt', 'iso639_3': 'udm', 'opus_code': 'udm'}
RUSSIAN = {'language_ru': 'русский', 'language_en': 'Russian', 'iso639_3': 'rus', 'opus_code': 'ru'}

hf_udmurt_datasets, hf_udmurt_errors = hf_datasets_for_language(UDMURT, limit_per_query=50)
print('HF candidates for Udmurt:', len(hf_udmurt_datasets))
if hf_udmurt_errors:
    print('HF language search errors:', hf_udmurt_errors[:3])
display(hf_dataset_brief_table(hf_udmurt_datasets).head(25))

save_artifact('lesson01_hf_udmurt_candidates.csv', hf_dataset_brief_table(hf_udmurt_datasets))

In [ ]:
hf_ru_udm_candidates, hf_ru_udm_errors = hf_datasets_for_pair(RUSSIAN, UDMURT, limit_per_query=50)
print('HF candidates for Russian-Udmurt pair:', len(hf_ru_udm_candidates))
if hf_ru_udm_errors:
    print('HF pair search errors:', hf_ru_udm_errors[:3])
display(hf_dataset_brief_table(hf_ru_udm_candidates).head(25))

save_artifact('lesson01_hf_ru_udm_pair_candidates.csv', hf_dataset_brief_table(hf_ru_udm_candidates))

## 5. Где заканчивается пайплайн и начинается агент

Первую часть задачи лучше решать обычным алгоритмом. У нас есть фиксированный список языков, заранее известные API, понятные поля ответа и воспроизводимые шаги обработки. Если нужно проверить конкретную языковую пару в OPUS/HF или скачать конкретный датасет, это проще, надежнее и дешевле сделать обычным кодом или руками.

Агенты становятся уместны сразу после этого: когда вход неформализован, источники заранее неизвестны, а набор решений нельзя полностью подготовить до запуска. Например: “найди все пригодные материалы для коми-пермяцкого, не перепутай его с коми-зырянским, отдели готовые датасеты от просто текстовых источников, оцени лицензионные риски и предложи, что проверить человеку”.

Такой поиск датасетов — недетерминированная исследовательская разведка: надо пройтись по выдаче, открыть страницы, понять, есть ли там датасет, найти ссылку на скачивание, проверить формат и решить, что отправить человеку на review. Ниже мы сначала соберем алгоритмическую таблицу по всем языкам, а потом для одного языка добавим аккуратного агента веб-разведки.

После OPUS/HF-примеров появляются три реальные развилки:

1. **Web discovery agent**: искать параллельные корпуса через поисковик, GitHub, страницы проектов и архивы; открывать страницы; искать скачиваемые ссылки; отправлять кандидатов на human review.
2. **HF card LLM classifier**: обкачать Hugging Face по тегам/поиску, получить много кандидатов вроде `fineweb`, `wikimedia/wikipedia` или широких multilingual-корпусов, а потом через LLM-классификацию оставить только настоящие параллельные корпуса для нужной пары.
3. **Monolingual extraction pipeline**: если большой корпус вроде FineWeb реально содержит нужный язык, отдельно строить не агент, а воспроизводимый пайплайн фильтрации, language identification, дедупликации и datacard.

## 6. Функции свертки источников в наблюдения

In [ ]:
def query_opus_for_language(opus_code):
    """Собирает сводку OPUS по моноязычным и русско-параллельным данным языка."""
    empty = {
        'opus_ru_parallel_pairs': 0,
        'opus_ru_parallel_documents': 0,
        'opus_ru_parallel_corpora': '',
        'opus_mono_pairs_or_segments': 0,
        'opus_mono_documents': 0,
        'opus_mono_corpora': '',
    }
    if not opus_code:
        return {'opus_checked': False, **empty}
    try:
        data = api_get(OPUS_API, {
            'source': 'ru',
            'target': opus_code,
            'preprocessing': 'xml',
            'version': 'latest',
        }, attempts=1, timeout=8)
    except Exception as exc:
        return {'opus_checked': False, 'opus_error': str(exc), **empty}
    corpora = data.get('corpora', [])
    parallel = [c for c in corpora if {c.get('source'), c.get('target')} == {'ru', opus_code}]
    mono = [c for c in corpora if c.get('source') == opus_code and not c.get('target')]
    return {
        'opus_checked': True,
        'opus_ru_parallel_pairs': sum(as_int(c.get('alignment_pairs')) for c in parallel),
        'opus_ru_parallel_documents': sum(as_int(c.get('documents')) for c in parallel),
        'opus_ru_parallel_corpora': '; '.join(f"{c.get('corpus')} ({c.get('alignment_pairs') or 0})" for c in parallel),
        'opus_mono_pairs_or_segments': sum(as_int(c.get('alignment_pairs')) for c in mono),
        'opus_mono_documents': sum(as_int(c.get('documents')) for c in mono),
        'opus_mono_corpora': '; '.join(f"{c.get('corpus')} ({c.get('alignment_pairs') or 0})" for c in mono),
    }

def query_huggingface_for_language(row):
    """Ищет датасеты на Hugging Face и собирает сводку вероятных ресурсов языка.

    HF-поиск обычно не дает точного количества документов или предложений.
    Поэтому сохраняем разведочные метаданные: число кандидатов, вероятные русско-
    параллельные датасеты, топ id датасетов, скачивания и категории размера из тегов.
    """
    language_tags = {
        f"language:{row.get('opus_code')}" if row.get('opus_code') else '',
        f"language:{row.get('iso639_3')}" if row.get('iso639_3') else '',
    }
    language_tags.discard('')
    search_terms = [row.get('language_en'), row.get('language_ru')]
    seen = {}
    attempted = 0
    successful = 0
    for tag in language_tags:
        attempted += 1
        try:
            results = api_get(HF_DATASETS_API, {'filter': tag, 'limit': 10}, attempts=2, timeout=20)
        except Exception:
            continue
        if not isinstance(results, list):
            continue
        successful += 1
        for dataset in results:
            dataset_id = dataset.get('id')
            if dataset_id:
                seen[dataset_id] = dataset

    for term in [x for x in search_terms if x]:
        attempted += 1
        try:
            results = api_get(HF_DATASETS_API, {'search': term, 'limit': 10}, attempts=2, timeout=20)
        except Exception:
            continue
        if not isinstance(results, list):
            continue
        successful += 1
        for dataset in results:
            dataset_id = dataset.get('id')
            if dataset_id:
                seen[dataset_id] = dataset

    lang_texts = [
        str(row.get('language_en', '')).lower(),
        str(row.get('language_ru', '')).lower(),
    ]

    def mentions_language_name(text, names):
        """Проверяет, встречается ли полное название языка как отдельная фраза."""
        for name in names:
            if not name:
                continue
            for part in re.split(r'\s*/\s*|\s+-\s+', name):
                part = part.strip()
                if len(part) >= 4 and re.search(rf'(?<![\w-]){re.escape(part)}(?![\w-])', text):
                    return True
        return False

    datasets = []
    for dataset in seen.values():
        tags = set(dataset.get('tags') or [])
        haystack = ' '.join([
            dataset.get('id', ''),
            dataset.get('description', '') or '',
            ' '.join(tags),
        ]).lower()
        tagged = bool(tags & language_tags)
        mentioned = mentions_language_name(haystack, lang_texts)
        if tagged or mentioned:
            datasets.append(dataset)

    def is_ru_parallel(dataset):
        """Эвристически определяет, похож ли HF-датасет на русско-параллельный."""
        tags = set(dataset.get('tags') or [])
        text = ' '.join([
            dataset.get('id', ''),
            dataset.get('description', '') or '',
            ' '.join(tags),
        ]).lower()
        return (
            'language:ru' in tags
            or 'russian' in text
            or 'рус' in text
            or '-rus-' in text
            or 'rus-' in text
        )

    def specificity_score(dataset):
        """Ставит языково-специфичные датасеты выше широких многоязычных коллекций."""
        tags = set(dataset.get('tags') or [])
        text = ' '.join([
            dataset.get('id', ''),
            dataset.get('description', '') or '',
        ]).lower()
        language_tag_count = sum(1 for tag in tags if tag.startswith('language:'))
        if mentions_language_name(text, lang_texts):
            return 2
        if language_tag_count <= 5:
            return 1
        return 0

    top = sorted(
        datasets,
        key=lambda d: (specificity_score(d), d.get('downloads') or 0),
        reverse=True,
    )[:5]
    size_categories = sorted({
        tag.replace('size_categories:', '')
        for dataset in datasets
        for tag in (dataset.get('tags') or [])
        if tag.startswith('size_categories:')
    })
    return {
        'hf_checked': successful > 0,
        'hf_query_attempts': attempted,
        'hf_query_successes': successful,
        'hf_dataset_count': len(datasets),
        'hf_ru_parallel_candidates': sum(1 for dataset in datasets if is_ru_parallel(dataset)),
        'hf_top_datasets': '; '.join(dataset.get('id', '') for dataset in top),
        'hf_downloads_sum': sum(int(dataset.get('downloads') or 0) for dataset in datasets),
        'hf_size_categories': '; '.join(size_categories),
        'hf_source_url': 'https://huggingface.co/datasets',
    }

## 7. Запуск пайплайна по всем языкам

In [ ]:
def scout_language(row):
    """Собирает все наблюдения по одному языку в одну сериализуемую строку."""
    observation = dict(row)
    observation.update(query_opus_for_language(row.get('opus_code')))
    observation.update(query_huggingface_for_language(row))
    observation['parallel_with_russian_source'] = 'https://opus.nlpl.eu/opusapi'
    observation['monolingual_source'] = 'OPUS monolingual rows; Hugging Face dataset search'
    observation['checked_at_utc'] = datetime.now(timezone.utc).strftime('%Y-%m-%d')
    return observation

def run_dataset_inventory_pipeline(languages):
    """Запускает воспроизводимый пайплайн инвентаризации датасетов."""
    result = {
        'sources': ['OPUS API', 'Hugging Face dataset API'],
        'languages_total': len(languages),
        'observations': [],
        'errors': [],
    }
    for i, lang in enumerate(languages, 1):
        print(f"[{i}/{len(languages)}] {lang['language_ru']}")
        try:
            result['observations'].append(scout_language(lang))
        except Exception as exc:
            result['errors'].append({'language_ru': lang['language_ru'], 'error': str(exc)})
    inventory = pd.DataFrame(result['observations'])
    inventory = inventory.sort_values(['family', 'branch', 'language_ru']).reset_index(drop=True)
    result['inventory'] = inventory
    result['summary'] = {
        'languages_total': len(languages),
        'languages_checked': len(inventory),
        'with_opus_ru_parallel': int((inventory['opus_ru_parallel_pairs'] > 0).sum()),
        'with_hf_candidates': int((inventory['hf_dataset_count'] > 0).sum()),
        'errors': len(result['errors']),
    }
    return result

pipeline_result = run_dataset_inventory_pipeline(LANGUAGES)
pipeline_result['summary']

In [ ]:
inventory = pipeline_result['inventory']
display(inventory.head(20))
display(inventory.groupby('family')[['opus_ru_parallel_pairs', 'opus_mono_pairs_or_segments']].sum().sort_values('opus_ru_parallel_pairs', ascending=False))

save_artifact('lesson01_language_dataset_inventory.csv', inventory)
save_artifact('lesson01_dataset_inventory_summary.json', json.dumps(pipeline_result['summary'], ensure_ascii=False, indent=2))

## 8. Google Sheets

На занятии можно открыть готовый Google Sheet и править его как общий рабочий артефакт:

https://docs.google.com/spreadsheets/d/1Qfr6JCB5CF-NLwQBODStqfhesrYw9tIVh2s_A6cg0d8

В Colab эта тетрадка сохраняет CSV в `/content/lowres_lab/lesson01_language_dataset_inventory.csv`. Его можно загрузить в Google Sheets или использовать как основу для обновления общей таблицы.

## 9. Как автоматизировать обновление

Разовый запуск полезен для старта, но карта датасетов быстро устаревает: в OPUS появляются новые релизы, в Hugging Face загружают корпуса, национальные проекты открывают новые таблицы, а часть ссылок ломается.

Для этого нужен регулярный фоновый пайплайн:

1. **Scheduler** запускает пайплайн по расписанию: например, раз в неделю или раз в месяц.
2. **Collector** заново обходит источники готовых датасетов: OPUS, Hugging Face, GitHub-релизы, национальные корпуса, каталоги открытых данных, архивы с опубликованными корпусами.
3. **State store** хранит предыдущий снимок таблицы: CSV в GitHub, Google Sheet, SQLite или маленький JSON.
4. **Diff checker** сравнивает старую и новую версии: новые языки, новые корпуса, рост/падение counts, ошибки API.
5. **Updater** обновляет Google Sheet только для безопасных полей: counts, даты проверки, ссылки на источники.
6. **Human review** получает спорные изменения: новый источник без понятной лицензии, резкое падение counts, объединение языков/вариантов, изменение классификации.

Где здесь веб-поиск? Его можно добавить отдельным ручным или полуавтоматическим слоем обнаружения: запросы вроде `"удмуртский корпус скачать"`, `"Udmurt dataset"`, `"site:github.com udmurt corpus"`, `"site:huggingface.co/datasets udmurt"`. Но веб-поиск лучше использовать как слой кандидатов, а не как источник финальных чисел. Найденные ссылки должны попадать в лист `candidates_for_review`, пока человек не подтвердит язык, лицензию, формат, объем и надежность источника.

Самый простой стек для курса:

- `scripts/build_language_dataset_inventory.py` лежит в GitHub;
- GitHub Actions запускает его по cron;
- скрипт сохраняет новый CSV;
- отдельный шаг через Google Sheets API обновляет таблицу;
- если diff большой или появились ошибки, workflow создает issue/комментарий для ручной проверки.

Colab для такого расписания не подходит: он хорош для занятия и ручного запуска, но не для надежного фонового мониторинга.

In [ ]:
def compare_inventory_snapshots(old_df, new_df):
    """Возвращает изменения по строкам между двумя снимками инвентаризации."""
    key = 'iso639_3'
    old = old_df.set_index(key)
    new = new_df.set_index(key)
    rows = []

    for code in sorted(set(old.index) | set(new.index)):
        if code not in old.index:
            rows.append({'iso639_3': code, 'change_type': 'new_language', 'needs_human_review': True})
            continue
        if code not in new.index:
            rows.append({'iso639_3': code, 'change_type': 'missing_language', 'needs_human_review': True})
            continue

        old_pairs = int(old.loc[code, 'opus_ru_parallel_pairs'])
        new_pairs = int(new.loc[code, 'opus_ru_parallel_pairs'])
        delta = new_pairs - old_pairs
        if delta != 0:
            rows.append({
                'iso639_3': code,
                'language_ru': new.loc[code, 'language_ru'],
                'change_type': 'parallel_count_changed',
                'old_pairs': old_pairs,
                'new_pairs': new_pairs,
                'delta': delta,
                'needs_human_review': abs(delta) > max(1000, old_pairs * 0.5),
            })

    return pd.DataFrame(rows)

# Мини-демо: имитируем, что через месяц OPUS нашел больше параллельных предложений для удмуртского.
old_snapshot = inventory.copy()
new_snapshot = inventory.copy()
new_snapshot.loc[new_snapshot['iso639_3'] == 'udm', 'opus_ru_parallel_pairs'] += 250

diff = compare_inventory_snapshots(old_snapshot, new_snapshot)
display(diff)
save_artifact('lesson01_inventory_diff_demo.csv', diff)

### Пример GitHub Actions расписания

```yaml
name: update-language-dataset-inventory

on:
  schedule:
    - cron: "0 6 1 * *"  # 1 числа каждого месяца
  workflow_dispatch:

jobs:
  update:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install pandas openpyxl requests
      - run: python scripts/build_language_dataset_inventory.py
      - name: Update Google Sheet
        run: python scripts/update_google_sheet.py
        env:
          GOOGLE_SERVICE_ACCOUNT_JSON: ${{ secrets.GOOGLE_SERVICE_ACCOUNT_JSON }}
          SPREADSHEET_ID: "1Qfr6JCB5CF-NLwQBODStqfhesrYw9tIVh2s_A6cg0d8"
```

В реальном проекте секреты Google API нельзя хранить в notebook. Их кладут в GitHub Secrets, Google Cloud Secret Manager или другой защищенный secret store.

## 10. Агент веб-разведки для одного языка

Теперь берем место, где агенты действительно в тему. OPUS/HF-пайплайн выше работает по известным API. Но реальная разведка датасетов часто начинается с неформализованного запроса:

> Найди открытые датасеты для удмуртского языка. Не считай Wikipedia готовым датасетом. Отделяй готовые датасеты от просто источников текстов. Если видишь GitHub, архив, корпус или страницу проекта, попробуй понять, можно ли скачать данные и что должен проверить человек.

Это уже не таблица по заранее известной схеме. Здесь агенту нужны:

- **state**: цель, язык, поисковые запросы, найденные страницы, кандидаты, ошибки;
- **tools**: web search, открытие страницы, извлечение ссылок, probe скачивания;
- **policy**: когда страницу считать кандидатом в датасет, когда отправлять на human review;
- **LLM-слой**: опционально, для планирования запросов и классификации неоднозначных страниц.

Ниже все ограничено специально для занятия: один язык, несколько запросов, несколько результатов, маленькие скачивания/probe, без автономного бесконечного браузинга.

In [ ]:
DISCOVERY_LANGUAGE = {
    'language_ru': 'удмуртский',
    'language_en': 'Udmurt',
    'iso639_3': 'udm',
    'opus_code': 'udm',
}

DISCOVERY_GOAL = '''
Найти дополнительные открытые датасеты или корпуса для удмуртского языка.
Не считать Wikipedia готовым датасетом. Отделять готовые датасеты от просто
источников текстов. Для каждого кандидата записать evidence, возможную лицензию,
тип данных и что должен проверить человек.
'''

MAX_QUERIES = 4
MAX_RESULTS_PER_QUERY = 4
MAX_PAGES_TO_OPEN = 10
MAX_DOWNLOAD_BYTES = 200_000

### 10.1. Tools: поиск, чтение страниц, проверка ссылок

In [ ]:
def normalize_ddg_url(href):
    """Достает настоящий URL из redirect-ссылки DuckDuckGo, если он там спрятан."""
    if not href:
        return ''
    parsed = urlparse(href)
    if 'duckduckgo.com' in parsed.netloc and parsed.path.startswith('/l/'):
        target = parse_qs(parsed.query).get('uddg', [''])[0]
        return unquote(target)
    return href

def web_search(query, max_results=5):
    """Ищет страницы через DuckDuckGo HTML и возвращает короткий список результатов.

    Это не официальный Google Search API, а учебный бесплатный инструмент.
    В production лучше использовать SerpAPI, Google Custom Search API, Brave Search API
    или другой легальный поисковый API с понятными лимитами.
    """
    url = 'https://duckduckgo.com/html/'
    try:
        r = requests.get(
            url,
            params={'q': query},
            headers={'User-Agent': 'Mozilla/5.0 lowres-course-dataset-discovery/1.0'},
            timeout=20,
        )
        r.raise_for_status()
    except Exception as exc:
        return [{'query': query, 'error': str(exc)}]

    if 'Unfortunately, bots use DuckDuckGo too' in r.text or 'anomaly-modal' in r.text:
        return [{
            'query': query,
            'error': 'DuckDuckGo challenge; no fallback used. Try later or replace web_search with an official search API.',
        }]

    soup = BeautifulSoup(r.text, 'html.parser')
    results = []
    for node in soup.select('a.result__a')[:max_results]:
        href = normalize_ddg_url(node.get('href'))
        title = ' '.join(node.get_text(' ', strip=True).split())
        snippet_node = node.find_parent('div', class_='result')
        snippet = ''
        if snippet_node:
            snippet = ' '.join(snippet_node.get_text(' ', strip=True).split())
        if href:
            results.append({
                'query': query,
                'title': title,
                'url': href,
                'snippet': snippet[:500],
                'search_mode': 'duckduckgo_html',
            })
    return results

def fetch_page(url, max_chars=8000):
    """Открывает HTML-страницу и возвращает текст, ссылки и технические метаданные."""
    try:
        r = requests.get(
            url,
            headers={'User-Agent': 'Mozilla/5.0 lowres-course-dataset-discovery/1.0'},
            timeout=20,
        )
        content_type = r.headers.get('Content-Type', '')
        status = r.status_code
        r.raise_for_status()
    except Exception as exc:
        return {
            'url': url,
            'status': 'error',
            'error': str(exc),
            'text': '',
            'links': [],
        }

    soup = BeautifulSoup(r.text, 'html.parser')
    for tag in soup(['script', 'style', 'noscript']):
        tag.decompose()
    text = ' '.join(soup.get_text(' ', strip=True).split())[:max_chars]
    links = []
    for a in soup.find_all('a', href=True):
        link_url = urljoin(url, a['href'])
        label = ' '.join(a.get_text(' ', strip=True).split())
        links.append({'url': link_url, 'label': label[:160]})

    return {
        'url': url,
        'status': status,
        'content_type': content_type,
        'text': text,
        'links': links[:80],
    }

DOWNLOAD_HINTS = (
    '.zip', '.tar.gz', '.tgz', '.csv', '.tsv', '.json', '.jsonl', '.txt',
    '.xml', '.conllu', '.parquet', '.xlsx', '.wav', '.mp3'
)

def likely_download_link(link):
    """Проверяет по URL и подписи, похожа ли ссылка на скачивание данных."""
    url = link.get('url', '').lower()
    label = link.get('label', '').lower()
    joined = f'{url} {label}'
    return (
        any(hint in url for hint in DOWNLOAD_HINTS)
        or 'download' in joined
        or 'raw.githubusercontent.com' in url
        or '/resolve/' in url
        or 'releases/download' in url
    )

def probe_download(url, max_bytes=MAX_DOWNLOAD_BYTES):
    """Аккуратно проверяет, доступна ли ссылка на данные, не скачивая большие файлы."""
    result = {'download_url': url}
    try:
        head = requests.head(
            url,
            allow_redirects=True,
            timeout=15,
            headers={'User-Agent': 'Mozilla/5.0 lowres-course-dataset-discovery/1.0'},
        )
        result.update({
            'head_status': head.status_code,
            'content_type': head.headers.get('Content-Type', ''),
            'content_length': head.headers.get('Content-Length', ''),
        })
        length = int(head.headers.get('Content-Length') or 0)
        if length and length > max_bytes:
            result['probe_status'] = 'too_large_for_class_demo'
            return result
    except Exception as exc:
        result['head_error'] = str(exc)

    try:
        get = requests.get(
            url,
            stream=True,
            timeout=20,
            headers={'User-Agent': 'Mozilla/5.0 lowres-course-dataset-discovery/1.0'},
        )
        chunk = next(get.iter_content(chunk_size=min(max_bytes, 4096)), b'')
        result.update({
            'get_status': get.status_code,
            'sample_bytes': len(chunk),
            'sample_text': chunk[:500].decode('utf-8', errors='replace'),
            'probe_status': 'sample_downloaded',
        })
    except Exception as exc:
        result['get_error'] = str(exc)
        result['probe_status'] = 'probe_failed'
    return result

### 10.2. Policy и опциональная LLM-классификация через OpenRouter

In [ ]:
DATASET_WORDS = [
    'dataset', 'corpus', 'parallel corpus', 'monolingual corpus', 'treebank',
    'download', 'github', 'huggingface', 'csv', 'jsonl', 'conllu', 'archive',
    'датасет', 'корпус', 'параллельный корпус', 'скачать', 'данные',
]

SOURCE_ONLY_WORDS = [
    'wikipedia', 'encyclopedia', 'news', 'article', 'blog', 'dictionary only',
    'википедия', 'новость', 'статья',
]

def heuristic_classify_page(page, language):
    """Классифицирует страницу простыми правилами, если LLM-ключа нет."""
    text = page.get('text', '').lower()
    url = page.get('url', '').lower()
    lang_hits = sum(token in text or token in url for token in [
        language['language_en'].lower(),
        language['language_ru'].lower(),
        language['iso639_3'].lower(),
    ])
    dataset_hits = sum(word in text or word in url for word in DATASET_WORDS)
    source_only_hits = sum(word in text or word in url for word in SOURCE_ONLY_WORDS)
    download_links = [link for link in page.get('links', []) if likely_download_link(link)]

    if dataset_hits >= 2 and lang_hits:
        label = 'dataset_candidate'
    elif dataset_hits >= 1 and download_links:
        label = 'possible_dataset_candidate'
    elif source_only_hits:
        label = 'source_or_reference_only'
    else:
        label = 'unclear'

    review = []
    if download_links:
        review.append('проверить скачиваемые ссылки и формат')
    if 'license' in text or 'лиценз' in text:
        review.append('проверить лицензию')
    else:
        review.append('лицензия не найдена автоматически')
    if source_only_hits:
        review.append('не считать источником финальных чисел без ручной проверки')

    return {
        'label': label,
        'confidence': 'medium' if label in {'dataset_candidate', 'source_or_reference_only'} else 'low',
        'dataset_type': 'unknown',
        'evidence': page.get('text', '')[:500],
        'needs_human_review': True,
        'human_review_note': '; '.join(review),
        'mode': 'heuristic',
    }

def call_openrouter_json(prompt, model='openai/gpt-oss-20b:free'):
    """Вызывает OpenRouter и ожидает JSON-ответ, если доступен OPENROUTER_API_KEY."""
    try:
        from google.colab import userdata
        api_key = userdata.get('OPENROUTER_API_KEY')
    except Exception:
        api_key = os.environ.get('OPENROUTER_API_KEY')
    if not api_key:
        return None

    response = requests.post(
        'https://openrouter.ai/api/v1/chat/completions',
        headers={
            'Authorization': f'Bearer {api_key}',
            'Content-Type': 'application/json',
        },
        json={
            'model': model,
            'messages': [
                {'role': 'system', 'content': 'Ты аккуратно классифицируешь страницы про языковые датасеты. Отвечай только JSON.'},
                {'role': 'user', 'content': prompt},
            ],
            'temperature': 0.1,
        },
        timeout=60,
    )
    response.raise_for_status()
    raw = response.json()['choices'][0]['message']['content']
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {'label': 'llm_unparsed', 'raw_answer': raw, 'needs_human_review': True}

def classify_page(page, language, goal):
    """Классифицирует страницу через LLM, а без ключа использует эвристики."""
    prompt = f"""Цель:
{goal}

Язык:
{json.dumps(language, ensure_ascii=False)}

Страница:
URL: {page.get('url')}
TEXT:
{page.get('text', '')[:5000]}

Верни JSON:
{{
  "label": "dataset_candidate | possible_dataset_candidate | source_or_reference_only | irrelevant | unclear",
  "confidence": "high | medium | low",
  "dataset_type": "parallel_text | monolingual_text | speech | dictionary | ocr | mixed | unknown",
  "evidence": "короткая цитата/пересказ признаков",
  "license_hint": "что видно про лицензию или пусто",
  "needs_human_review": true,
  "human_review_note": "что именно проверить человеку"
}}
"""
    llm_answer = call_openrouter_json(prompt)
    if llm_answer is not None:
        llm_answer['mode'] = 'openrouter'
        return llm_answer
    return heuristic_classify_page(page, language)

### 10.3. Сценарий 2: LLM-классификатор HF-карточек

Перед веб-поиском можно сделать более контролируемую LLM-задачу: взять кандидатов из HF API и отсеять широкие корпуса, которые только содержат языковой тег, но не являются параллельным корпусом для нужной пары.

Например, `fineweb`, `wikimedia/wikipedia`, `GlotCC` или `DCAD` могут быть полезны для моноязычного сценария, но они не становятся русско-удмуртским параллельным корпусом только потому, что где-то содержат `language:udm` или слово `Udmurt`.
Это не агент: здесь нет планирования, выбора tools и изменения стратегии. Это LLM-классификация заранее собранного списка.

In [ ]:
def needs_openrouter_classification(dataset):
    """Возвращает честную заглушку, если OpenRouter-ключ не подключен."""
    tags = dataset.get('tags') or []
    return {
        'dataset_id': dataset.get('id'),
        'label': 'needs_openrouter_key',
        'confidence': 'none',
        'reason': 'классификация параллельности отключена: нет OPENROUTER_API_KEY',
        'evidence': dataset_search_text(dataset)[:500],
        'needs_human_review': True,
        'downloads': dataset.get('downloads'),
        'language_tags': '; '.join(tag for tag in tags if tag.startswith('language:')),
        'task_tags': '; '.join(tag for tag in tags if tag.startswith('task_categories:')),
        'url': hf_dataset_page_url(dataset.get('id', '')),
        'mode': 'no_llm',
    }

def classify_hf_parallel_candidate(dataset, left_language, right_language):
    """Классифицирует HF-карточку через OpenRouter и возвращает JSON-решение."""
    compact_text = dataset_search_text(dataset)
    prompt = f"""Нужно понять, является ли Hugging Face dataset настоящим параллельным корпусом для языковой пары.

Левый язык:
{json.dumps(left_language, ensure_ascii=False)}

Правый язык:
{json.dumps(right_language, ensure_ascii=False)}

Dataset object:
{json.dumps({
    'id': dataset.get('id'),
    'description': dataset.get('description'),
    'tags': dataset.get('tags'),
    'metadata_text': compact_text[:3000],
    'downloads': dataset.get('downloads'),
    'likes': dataset.get('likes'),
}, ensure_ascii=False)[:5000]}

Верни JSON:
{{
  "dataset_id": "...",
  "label": "parallel_corpus | parallel_candidate_needs_review | not_parallel_broad_multilingual | language_related_not_parallel | irrelevant_or_unclear",
  "confidence": "high | medium | low",
  "reason": "почему",
  "evidence": "короткие признаки из карточки",
  "needs_human_review": true
}}
"""
    llm_answer = call_openrouter_json(prompt)
    if llm_answer is not None:
        llm_answer['dataset_id'] = llm_answer.get('dataset_id') or dataset.get('id')
        llm_answer['url'] = hf_dataset_page_url(dataset.get('id', ''))
        llm_answer['mode'] = 'openrouter'
        return llm_answer
    return needs_openrouter_classification(dataset)

def run_hf_card_review_classifier(datasets, left_language, right_language, max_cards=30):
    """Проходит по HF-кандидатам и классифицирует параллельность через LLM."""
    state = {
        'goal': 'отфильтровать HF-кандидаты до параллельных корпусов для языковой пары',
        'left_language': left_language,
        'right_language': right_language,
        'cards_total': len(datasets),
        'cards_reviewed': 0,
        'decisions': [],
    }
    for dataset in datasets[:max_cards]:
        decision = classify_hf_parallel_candidate(dataset, left_language, right_language)
        state['decisions'].append(decision)
        state['cards_reviewed'] += 1
    return state

hf_review_state = run_hf_card_review_classifier(hf_ru_udm_candidates, RUSSIAN, UDMURT, max_cards=30)
hf_review_df = pd.DataFrame(hf_review_state['decisions'])
display(hf_review_df.sort_values(['label', 'confidence']).head(30))
save_artifact('lesson01_hf_parallel_review_classifier.csv', hf_review_df)

In [ ]:
if len(hf_review_df):
    display(hf_review_df['label'].value_counts().reset_index(name='count').rename(columns={'index': 'label'}))
    display(hf_review_df[hf_review_df['label'].isin(['parallel_corpus', 'parallel_candidate_needs_review'])][[
        'dataset_id', 'label', 'confidence', 'reason', 'url'
    ]])

Это не агентная задача: вход уже собран алгоритмом, tools не выбираются, стратегия не меняется. Но это хороший пример LLM-классификации: модель читает metadata карточек и возвращает JSON с решением, является ли карточка параллельным корпусом или широким многоязычным/моноязычным ресурсом.

### 10.4. Сценарий 1: агент руками для web discovery

In [ ]:
def plan_search_queries(language, goal):
    """Планирует несколько поисковых запросов для одного языка."""
    prompt = f"""Составь до 4 поисковых запросов для поиска датасетов языка.
Язык: {json.dumps(language, ensure_ascii=False)}
Цель: {goal}
Верни JSON: {{"queries": ["..."]}}
"""
    llm_answer = call_openrouter_json(prompt)
    if llm_answer and isinstance(llm_answer.get('queries'), list):
        return llm_answer['queries'][:MAX_QUERIES]
    return [
        f"{language['language_en']} dataset corpus",
        f"{language['language_en']} Russian parallel corpus",
        f"{language['language_en']} language GitHub corpus",
        f"{language['language_ru']} корпус скачать датасет",
    ][:MAX_QUERIES]

def run_manual_discovery_agent(language, goal):
    """Запускает маленького агента веб-разведки без фреймворка."""
    state = {
        'goal': goal,
        'language': language,
        'queries': [],
        'search_results': [],
        'pages': [],
        'candidates': [],
        'download_probes': [],
        'errors': [],
    }
    state['queries'] = plan_search_queries(language, goal)

    seen_urls = set()
    for query in state['queries']:
        print('search:', query)
        results = web_search(query, max_results=MAX_RESULTS_PER_QUERY)
        state['search_results'].extend(results)
        for item in results:
            url = item.get('url')
            if not url or url in seen_urls or item.get('error'):
                continue
            seen_urls.add(url)
            if len(state['pages']) >= MAX_PAGES_TO_OPEN:
                break
            page = fetch_page(url)
            state['pages'].append(page)
            if page.get('status') == 'error':
                state['errors'].append({'url': url, 'error': page.get('error')})
                continue
            classification = classify_page(page, language, goal)
            candidate = {
                'language_ru': language['language_ru'],
                'language_en': language['language_en'],
                'url': url,
                'title': item.get('title', ''),
                'query': query,
                **classification,
            }
            candidate_links = [link for link in page.get('links', []) if likely_download_link(link)][:3]
            candidate['download_link_count'] = len(candidate_links)
            candidate['download_links'] = '; '.join(link['url'] for link in candidate_links)
            state['candidates'].append(candidate)
            for link in candidate_links[:1]:
                state['download_probes'].append({
                    'page_url': url,
                    **probe_download(link['url']),
                })
    return state

manual_agent_state = run_manual_discovery_agent(DISCOVERY_LANGUAGE, DISCOVERY_GOAL)
print('pages opened:', len(manual_agent_state['pages']))
print('candidates:', len(manual_agent_state['candidates']))
print('download probes:', len(manual_agent_state['download_probes']))

In [ ]:
manual_candidates = pd.DataFrame(manual_agent_state['candidates'])
if len(manual_candidates):
    display(manual_candidates[[
        'language_ru', 'label', 'confidence', 'dataset_type',
        'title', 'url', 'download_link_count', 'human_review_note', 'mode'
    ]].head(20))
    save_artifact('lesson01_manual_agent_candidates.csv', manual_candidates)

manual_probes = pd.DataFrame(manual_agent_state['download_probes'])
if len(manual_probes):
    display(manual_probes.head(10))
    save_artifact('lesson01_manual_agent_download_probes.csv', manual_probes)

Что здесь агентского:

- вход свободный, а не набор параметров API;
- агент сам превращает цель в поисковые запросы;
- список страниц заранее неизвестен;
- для каждой страницы надо принять неоднозначное решение: датасет, источник текстов, мусор или кандидат на review;
- если на странице есть скачиваемые ссылки, агент сам выбирает, что аккуратно проверить;
- результат не финальная истина, а очередь для human review.

OpenRouter здесь отвечает только за LLM-решения. Он не гуглит и не скачивает сам. Поиск, чтение страниц и probe скачивания — это tools, которые мы написали отдельно.

### 10.5. Та же web-разведка в LangGraph

In [ ]:
class DiscoveryState(TypedDict, total=False):
    goal: str
    language: Dict[str, Any]
    queries: List[str]
    search_results: List[Dict[str, Any]]
    pages: List[Dict[str, Any]]
    candidates: List[Dict[str, Any]]
    download_probes: List[Dict[str, Any]]
    errors: List[Dict[str, Any]]
    report: Dict[str, Any]

def plan_node(state: DiscoveryState):
    """Планирует поисковые запросы из свободной цели и языка."""
    return {'queries': plan_search_queries(state['language'], state['goal'])}

def search_node(state: DiscoveryState):
    """Запускает web_search по всем запланированным запросам."""
    results = []
    for query in state['queries'][:MAX_QUERIES]:
        results.extend(web_search(query, max_results=MAX_RESULTS_PER_QUERY))
    return {'search_results': results}

def inspect_node(state: DiscoveryState):
    """Открывает найденные страницы, классифицирует их и проверяет ссылки на данные."""
    pages = []
    candidates = []
    probes = []
    errors = []
    seen = set()
    for item in state['search_results']:
        url = item.get('url')
        if not url or url in seen or item.get('error'):
            continue
        seen.add(url)
        if len(pages) >= MAX_PAGES_TO_OPEN:
            break
        page = fetch_page(url)
        pages.append(page)
        if page.get('status') == 'error':
            errors.append({'url': url, 'error': page.get('error')})
            continue
        classification = classify_page(page, state['language'], state['goal'])
        links = [link for link in page.get('links', []) if likely_download_link(link)][:3]
        candidates.append({
            'language_ru': state['language']['language_ru'],
            'language_en': state['language']['language_en'],
            'url': url,
            'title': item.get('title', ''),
            'query': item.get('query', ''),
            **classification,
            'download_link_count': len(links),
            'download_links': '; '.join(link['url'] for link in links),
        })
        for link in links[:1]:
            probes.append({'page_url': url, **probe_download(link['url'])})
    return {'pages': pages, 'candidates': candidates, 'download_probes': probes, 'errors': errors}

def report_node(state: DiscoveryState):
    """Собирает короткий отчет по результатам разведки."""
    candidates = state.get('candidates', [])
    useful = [c for c in candidates if c.get('label') in {'dataset_candidate', 'possible_dataset_candidate'}]
    return {'report': {
        'language_ru': state['language']['language_ru'],
        'queries': state.get('queries', []),
        'search_results': len(state.get('search_results', [])),
        'pages_opened': len(state.get('pages', [])),
        'candidates_total': len(candidates),
        'dataset_candidates': len(useful),
        'download_probes': len(state.get('download_probes', [])),
        'errors': len(state.get('errors', [])),
    }}

discovery_graph = StateGraph(DiscoveryState)
discovery_graph.add_node('plan', plan_node)
discovery_graph.add_node('search', search_node)
discovery_graph.add_node('inspect', inspect_node)
discovery_graph.add_node('report', report_node)
discovery_graph.set_entry_point('plan')
discovery_graph.add_edge('plan', 'search')
discovery_graph.add_edge('search', 'inspect')
discovery_graph.add_edge('inspect', 'report')
discovery_graph.add_edge('report', END)

dataset_discovery_agent = discovery_graph.compile()
graph_agent_state = dataset_discovery_agent.invoke({
    'language': DISCOVERY_LANGUAGE,
    'goal': DISCOVERY_GOAL,
})
graph_agent_state['report']

In [ ]:
graph_candidates = pd.DataFrame(graph_agent_state['candidates'])
if len(graph_candidates):
    display(graph_candidates[[
        'language_ru', 'label', 'confidence', 'dataset_type',
        'title', 'url', 'download_link_count', 'human_review_note', 'mode'
    ]].head(20))
    save_artifact('lesson01_langgraph_agent_candidates.csv', graph_candidates)

graph_probes = pd.DataFrame(graph_agent_state['download_probes'])
if len(graph_probes):
    display(graph_probes.head(10))
    save_artifact('lesson01_langgraph_agent_download_probes.csv', graph_probes)

LangGraph не делает агента умнее сам по себе. Его смысл здесь в другом: он явно фиксирует архитектуру и состояние.

В plain Python версии мы сами держали порядок шагов в цикле. В LangGraph версии те же решения разложены по узлам:

1. `plan`: превратить свободную цель в поисковые запросы;
2. `search`: собрать выдачу;
3. `inspect`: открыть страницы, классифицировать, проверить ссылки;
4. `report`: собрать краткий отчет.

Дальше этот граф можно расширять: добавить условный повтор поиска, отдельного license critic, human-in-the-loop узел, запись в Google Sheet, ограничение бюджета и сохранение state между запусками.

### 10.6. Сценарий 3: моноязычные данные из больших корпусов

Это уже скорее не агент, а отдельный воспроизводимый pipeline. Если HF-card review показывает, что `fineweb`, `GlotCC`, `DCAD` или другой большой корпус содержит нужный язык, дальше задача меняется:

1. скачать или стримить только нужный shard/конфигурацию;
2. прогнать language identification;
3. отфильтровать строки нужного языка;
4. убрать дубликаты, boilerplate и слишком короткие/битые тексты;
5. посчитать объем, примеры, домены, риски;
6. оформить datacard и список ограничений.

Агент может помочь спроектировать такой pipeline для конкретного корпуса или проверить datacard, но сам extraction лучше делать обычным кодом: так результат воспроизводим и его можно перепроверить.

## Вопросы для отчета

1. Какие языковые семьи в таблице оказываются лучше всего покрыты параллельными данными с русским?
2. Где есть Википедия, но почти нет параллельных данных?
3. Где OPUS показывает нули: это значит “данных нет” или “мы не нашли правильный код/источник”?
4. Какие источники надо добавить следующими: национальные корпуса, сайты СМИ, библиотеки, архивы, Hugging Face, GitHub?
5. Какие результаты HF-поиска выглядят полезными, но требуют ручной проверки?
6. Какие веб-поисковые запросы вы бы добавили для своего языка?
7. Какие поля можно обновлять автоматически, а какие требуют human review?
8. Как часто стоит запускать фоновое обновление для такой таблицы и почему?
9. Чем веб-разведка датасетов отличается от OPUS/HF-пайплайна?
10. Какие решения агента в этом примере обязательно должен проверить человек?